# ACE ML Models - Model Registry

This notebook trains ML models for the ACE Intelligence Agent:
- **Service Request Volume Forecasting** - Predict future monthly service request volume
- **Member Churn Prediction** - Classify members at risk of non-renewal or cancellation
- **Response Success Prediction** - Predict service fulfillment success based on conditions

All models are registered to Snowflake Model Registry and can be added as tools to the Intelligence Agent.

## Prerequisites

**Required Packages** (configured automatically):
- `snowflake-ml-python`
- `scikit-learn`
- `xgboost`
- `matplotlib`

**Database Context:**
- **Database:** AAA_INTELLIGENCE  
- **Schema:** ANALYTICS  
- **Warehouse:** AAA_WH

**Note:** This notebook uses Snowflake Model Registry. Ensure you have appropriate permissions to create and register models.

## Import Required Packages

In [ ]:
# Import Python packages
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Import Snowpark
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
import snowflake.snowpark.types as T
from snowflake.snowpark import Window

# Import Snowpark ML
from snowflake.ml.modeling.preprocessing import StandardScaler, OneHotEncoder
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.linear_model import LinearRegression, LogisticRegression
from snowflake.ml.modeling.ensemble import RandomForestClassifier
from snowflake.ml.modeling.metrics import mean_squared_error, mean_absolute_error, accuracy_score, roc_auc_score
from snowflake.ml.registry import Registry

print("✅ Packages imported successfully")

## Connect to Snowflake

Get active session and set context to ACE database.

In [ ]:
# Get active Snowflake session
session = get_active_session()

# Set context
session.use_database('AAA_INTELLIGENCE')
session.use_schema('ANALYTICS')
session.use_warehouse('AAA_WH')

print(f"✅ Connected - Role: {session.get_current_role()}")
print(f"   Warehouse: {session.get_current_warehouse()}")
print(f"   Database.Schema: {session.get_fully_qualified_current_schema()}")

---
# MODEL 1: Service Request Volume Forecasting

Predict future monthly service request volume based on historical patterns, seasonality, and service types.

### Prepare Service Volume Training Data

In [ ]:
# Get monthly service request volume data with features
service_volume_df = session.sql("""
SELECT
    DATE_TRUNC('month', request_timestamp)::DATE AS service_month,
    MONTH(request_timestamp) AS month_num,
    YEAR(request_timestamp) AS year_num,
    COUNT(DISTINCT service_id)::FLOAT AS total_service_requests,
    COUNT(DISTINCT member_id)::FLOAT AS unique_members,
    COUNT(DISTINCT vehicle_id)::FLOAT AS unique_vehicles,
    AVG(CASE WHEN priority = 'HIGH' THEN 1.0 ELSE 0.0 END)::FLOAT AS high_priority_ratio,
    COUNT(DISTINCT CASE WHEN service_type = 'TOWING' THEN service_id END)::FLOAT AS towing_count,
    COUNT(DISTINCT CASE WHEN service_type = 'TIRE_CHANGE' THEN service_id END)::FLOAT AS tire_count,
    COUNT(DISTINCT CASE WHEN service_type = 'BATTERY_JUMP' THEN service_id END)::FLOAT AS battery_count,
    AVG(CASE WHEN weather_condition IN ('RAIN', 'SNOW', 'ICE') THEN 1.0 ELSE 0.0 END)::FLOAT AS bad_weather_ratio
FROM RAW.SERVICE_REQUESTS
WHERE request_timestamp >= DATEADD('month', -36, CURRENT_DATE())
  AND request_timestamp < CURRENT_DATE()
GROUP BY DATE_TRUNC('month', request_timestamp), MONTH(request_timestamp), YEAR(request_timestamp)
ORDER BY service_month
""")

print(f"Service volume data: {service_volume_df.count()} months")
service_volume_df.show(5)

### Split Data and Train Service Volume Model

In [ ]:
# Train/test split (last 6 months for testing)
train_service_volume = service_volume_df.filter(F.col("SERVICE_MONTH") < F.dateadd("month", F.lit(-6), F.current_date()))
test_service_volume = service_volume_df.filter(F.col("SERVICE_MONTH") >= F.dateadd("month", F.lit(-6), F.current_date()))

# Drop SERVICE_MONTH (DATE type not supported in pipeline)
train_service_volume = train_service_volume.drop("SERVICE_MONTH")
test_service_volume = test_service_volume.drop("SERVICE_MONTH")

# Create pipeline with scaling and regression
service_volume_pipeline = Pipeline([
    ("Scaler", StandardScaler(
        input_cols=["MONTH_NUM", "UNIQUE_MEMBERS", "UNIQUE_VEHICLES", "HIGH_PRIORITY_RATIO", 
                   "TOWING_COUNT", "TIRE_COUNT", "BATTERY_COUNT", "BAD_WEATHER_RATIO"],
        output_cols=["MONTH_NUM_SCALED", "UNIQUE_MEMBERS_SCALED", "UNIQUE_VEHICLES_SCALED", 
                    "HIGH_PRIORITY_RATIO_SCALED", "TOWING_COUNT_SCALED", "TIRE_COUNT_SCALED", 
                    "BATTERY_COUNT_SCALED", "BAD_WEATHER_RATIO_SCALED"]
    )),
    ("LinearRegression", LinearRegression(
        label_cols=["TOTAL_SERVICE_REQUESTS"],
        output_cols=["PREDICTED_SERVICE_REQUESTS"]
    ))
])

# Train model
service_volume_pipeline.fit(train_service_volume)
print("✅ Service request volume forecasting model trained")

### Evaluate and Register Service Volume Model

In [ ]:
# Make predictions on test set
test_predictions = service_volume_pipeline.predict(test_service_volume)

# Calculate metrics
mae = mean_absolute_error(df=test_predictions, y_true_col_names="TOTAL_SERVICE_REQUESTS", y_pred_col_names="PREDICTED_SERVICE_REQUESTS")
mse = mean_squared_error(df=test_predictions, y_true_col_names="TOTAL_SERVICE_REQUESTS", y_pred_col_names="PREDICTED_SERVICE_REQUESTS")
rmse = mse ** 0.5

metrics = {"mae": round(mae, 2), "rmse": round(rmse, 2)}
print(f"Model metrics: {metrics}")

# Register model
reg = Registry(session)
reg.log_model(
    model=service_volume_pipeline,
    model_name="SERVICE_VOLUME_PREDICTOR",
    version_name="V1",
    comment="Predicts monthly service request volume based on historical patterns, service mix, and weather conditions using Linear Regression",
    metrics=metrics
)

print("✅ Service volume model registered to Model Registry as SERVICE_VOLUME_PREDICTOR")

---
# MODEL 2: Member Churn Prediction

Classify members as likely to churn (non-renewal or cancellation) based on service usage patterns and satisfaction.

### Prepare Churn Training Data

In [ ]:
# Get member features for churn prediction
churn_df = session.sql("""
SELECT
    m.member_id,
    m.membership_level,
    m.risk_score::FLOAT AS risk_score,
    m.lifetime_value::FLOAT AS lifetime_value,
    DATEDIFF('day', m.membership_start_date, CURRENT_DATE())::FLOAT AS membership_days,
    DATEDIFF('day', CURRENT_DATE(), m.membership_renewal_date)::FLOAT AS days_to_renewal,
    m.is_auto_renew::BOOLEAN AS is_auto_renew,
    -- Service usage patterns (last 6 months)
    COUNT(DISTINCT sr.service_id)::FLOAT AS service_requests_6m,
    COUNT(DISTINCT CASE WHEN sr.service_type = 'TOWING' THEN sr.service_id END)::FLOAT AS towing_requests_6m,
    -- Service fulfillment metrics
    AVG(sf.response_time_minutes)::FLOAT AS avg_response_time,
    AVG(sf.member_satisfaction_score)::FLOAT AS avg_satisfaction_score,
    COUNT(DISTINCT CASE WHEN sf.member_satisfaction_score <= 2 THEN sf.service_id END)::FLOAT AS low_satisfaction_count,
    -- Transaction history
    COUNT(DISTINCT mt.transaction_id)::FLOAT AS total_transactions,
    SUM(CASE WHEN mt.transaction_type = 'RENEWAL' THEN 1 ELSE 0 END)::FLOAT AS renewal_count,
    -- Predictive scores
    AVG(ps.churn_risk_score)::FLOAT AS avg_churn_risk_score,
    -- Target: Is churned (membership status)
    (m.membership_status = 'CANCELLED')::BOOLEAN AS is_churned
FROM RAW.MEMBERS m
LEFT JOIN RAW.SERVICE_REQUESTS sr ON m.member_id = sr.member_id 
    AND sr.request_timestamp >= DATEADD('month', -6, CURRENT_DATE())
LEFT JOIN RAW.SERVICE_FULFILLMENT sf ON sr.service_id = sf.service_id
LEFT JOIN RAW.MEMBER_TRANSACTIONS mt ON m.member_id = mt.member_id
LEFT JOIN RAW.PREDICTIVE_SCORES ps ON m.member_id = ps.member_id
WHERE m.membership_status IN ('ACTIVE', 'CANCELLED')
  AND m.membership_start_date <= DATEADD('month', -12, CURRENT_DATE()) -- At least 1 year old
GROUP BY m.member_id, m.membership_level, m.risk_score, m.lifetime_value, 
         m.membership_start_date, m.membership_renewal_date, m.is_auto_renew, m.membership_status
HAVING COUNT(DISTINCT sr.service_id) > 0 OR COUNT(DISTINCT mt.transaction_id) > 0
LIMIT 10000  -- Limit for faster training
""")

print(f"Churn data: {churn_df.count()} members")
churn_df.show(5)

### Train Churn Classification Model

In [ ]:
# Train/test split (80/20)
train_churn, test_churn = churn_df.random_split([0.8, 0.2], seed=42)

# Drop MEMBER_ID
train_churn = train_churn.drop("MEMBER_ID")
test_churn = test_churn.drop("MEMBER_ID")

# Create pipeline with preprocessing and classification
churn_pipeline = Pipeline([
    ("Encoder", OneHotEncoder(
        input_cols=["MEMBERSHIP_LEVEL"],
        output_cols=["MEMBERSHIP_LEVEL_ENCODED"],
        drop_input_cols=True,  # Drop original string columns after encoding
        handle_unknown="ignore"
    )),
    ("Classifier", RandomForestClassifier(
        label_cols=["IS_CHURNED"],
        output_cols=["CHURN_PREDICTION"],
        n_estimators=100,
        max_depth=10,
        random_state=42
    ))
])

# Train model
churn_pipeline.fit(train_churn)
print("✅ Member churn classification model trained")

### Evaluate and Register Churn Model

In [ ]:
# Make predictions
churn_predictions = churn_pipeline.predict(test_churn)

# Calculate metrics
accuracy = accuracy_score(df=churn_predictions, y_true_col_names="IS_CHURNED", y_pred_col_names="CHURN_PREDICTION")
churn_metrics = {"accuracy": round(accuracy, 4)}
print(f"Churn model metrics: {churn_metrics}")

# Register model
reg.log_model(
    model=churn_pipeline,
    model_name="MEMBER_CHURN_PREDICTOR",
    version_name="V1",
    comment="Predicts member churn probability using Random Forest based on service usage patterns and satisfaction metrics",
    metrics=churn_metrics
)

print("✅ Churn model registered to Model Registry as MEMBER_CHURN_PREDICTOR")

---
# MODEL 3: Response Success Prediction

Predict which service requests are likely to be completed successfully within SLA based on conditions and resources.

### Prepare Response Success Data

In [ ]:
# Get service request features for success prediction
response_success_df = session.sql("""
SELECT
    sr.service_id,
    sr.service_type,
    sr.service_category,
    sr.priority,
    sr.location_type,
    sr.weather_condition,
    sr.temperature_f::FLOAT AS temperature_f,
    sr.channel,
    -- Time features
    HOUR(sr.request_timestamp)::INT AS request_hour,
    DAYOFWEEK(sr.request_timestamp)::INT AS request_dow,
    -- Regional features
    reg.average_response_time_minutes::FLOAT AS region_avg_response,
    reg.active_technicians::FLOAT AS region_technicians,
    reg.active_trucks::FLOAT AS region_trucks,
    -- Member features
    m.membership_level,
    v.vehicle_type,
    -- Technician features
    t.certification_level,
    t.average_response_time_minutes::FLOAT AS tech_avg_response,
    -- Success criteria: Completed within regional SLA and high satisfaction
    (sf.service_outcome = 'COMPLETED' 
     AND sf.response_time_minutes <= reg.target_response_time_minutes
     AND (sf.member_satisfaction_score >= 4 OR sf.member_satisfaction_score IS NULL))::BOOLEAN AS response_successful
FROM RAW.SERVICE_REQUESTS sr
JOIN RAW.SERVICE_FULFILLMENT sf ON sr.service_id = sf.service_id
JOIN RAW.SERVICE_TECHNICIANS t ON sf.technician_id = t.technician_id
JOIN RAW.MEMBERS m ON sr.member_id = m.member_id
LEFT JOIN RAW.VEHICLES v ON sr.vehicle_id = v.vehicle_id
LEFT JOIN RAW.SERVICE_REGIONS reg ON t.service_region = reg.region_name
WHERE sr.request_timestamp >= DATEADD('month', -12, CURRENT_DATE())
  AND sf.completion_timestamp IS NOT NULL
""")

print(f"Response success data: {response_success_df.count()} service requests")
response_success_df.show(5)

### Train Response Success Model

In [ ]:
# Split data
train_response, test_response = response_success_df.random_split([0.8, 0.2], seed=42)

# Drop SERVICE_ID
train_response = train_response.drop("SERVICE_ID")
test_response = test_response.drop("SERVICE_ID")

# Create pipeline
response_pipeline = Pipeline([
    ("Encoder", OneHotEncoder(
        input_cols=["SERVICE_TYPE", "SERVICE_CATEGORY", "PRIORITY", "LOCATION_TYPE", 
                   "WEATHER_CONDITION", "CHANNEL", "MEMBERSHIP_LEVEL", "VEHICLE_TYPE", 
                   "CERTIFICATION_LEVEL"],
        output_cols=["SERVICE_TYPE_ENC", "SERVICE_CATEGORY_ENC", "PRIORITY_ENC", "LOCATION_TYPE_ENC",
                    "WEATHER_CONDITION_ENC", "CHANNEL_ENC", "MEMBERSHIP_LEVEL_ENC", "VEHICLE_TYPE_ENC",
                    "CERTIFICATION_LEVEL_ENC"],
        drop_input_cols=True,
        handle_unknown="ignore"
    )),
    ("Classifier", LogisticRegression(
        label_cols=["RESPONSE_SUCCESSFUL"],
        output_cols=["SUCCESS_PREDICTION"]
    ))
])

# Train
response_pipeline.fit(train_response)
print("✅ Response success model trained")

### Evaluate and Register Response Success Model

In [ ]:
# Predict on test set
response_predictions = response_pipeline.predict(test_response)

# Calculate accuracy
response_accuracy = accuracy_score(df=response_predictions, 
                                 y_true_col_names="RESPONSE_SUCCESSFUL",
                                 y_pred_col_names="SUCCESS_PREDICTION")
response_metrics = {"accuracy": round(response_accuracy, 4)}
print(f"Response success model metrics: {response_metrics}")

# Register model
reg.log_model(
    model=response_pipeline,
    model_name="RESPONSE_SUCCESS_PREDICTOR",
    version_name="V1",
    comment="Predicts roadside service success (completed within SLA with high satisfaction) using Logistic Regression based on conditions, resources, and technician skills",
    metrics=response_metrics
)

print("✅ Response success model registered to Model Registry as RESPONSE_SUCCESS_PREDICTOR")

---
# Verify Models in Registry

In [ ]:
# Show all models in the registry
print("Models in registry:")
reg.show_models()

# Show versions for service volume model
print("\nService Volume Predictor versions:")
reg.get_model("SERVICE_VOLUME_PREDICTOR").show_versions()

# Show versions for churn model  
print("\nMember Churn Predictor versions:")
reg.get_model("MEMBER_CHURN_PREDICTOR").show_versions()

# Show versions for response success model
print("\nResponse Success Predictor versions:")
reg.get_model("RESPONSE_SUCCESS_PREDICTOR").show_versions()

print("\n✅ All models registered and ready to add to Intelligence Agent")

---
# Test Model Inference

Test calling each model to make predictions.

In [ ]:
# Test service volume forecast on recent data
service_model = reg.get_model("SERVICE_VOLUME_PREDICTOR").default
recent_service = service_volume_df.limit(3).drop("SERVICE_MONTH")
service_preds = service_model.run(recent_service, function_name="predict")
print("Service Volume predictions:")
service_preds.select("TOTAL_SERVICE_REQUESTS", "PREDICTED_SERVICE_REQUESTS").show()

# Test churn prediction on sample members
churn_model = reg.get_model("MEMBER_CHURN_PREDICTOR").default
sample_members = churn_df.limit(5).drop("MEMBER_ID")
churn_preds = churn_model.run(sample_members, function_name="predict")
print("\nChurn predictions:")
churn_preds.select("IS_CHURNED", "CHURN_PREDICTION").show()

# Test response success prediction
response_model = reg.get_model("RESPONSE_SUCCESS_PREDICTOR").default
sample_responses = response_success_df.limit(5).drop("SERVICE_ID")
response_preds = response_model.run(sample_responses, function_name="predict")
print("\nResponse Success predictions:")
response_preds.select("RESPONSE_SUCCESSFUL", "SUCCESS_PREDICTION").show()

print("\n✅ All models tested successfully!")

---
# Next Steps

## Add Models to Intelligence Agent

**Option 1: Using the SQL Script (Easiest)**
Run `sql/agent/08_create_intelligence_agent.sql` which automatically configures all 3 ML models.

**Option 2: Manual Configuration in Snowsight**
1. In Snowsight → AI & ML → Agents → AAA_INTELLIGENCE_AGENT
2. Go to Tools → + Add → Function
3. Add each model wrapper procedure:
   - **PREDICT_SERVICE_VOLUME** (from `sql/ml/07_create_model_wrapper_functions.sql`)
   - **PREDICT_MEMBER_CHURN** (from `sql/ml/07_create_model_wrapper_functions.sql`)
   - **PREDICT_RESPONSE_SUCCESS** (from `sql/ml/07_create_model_wrapper_functions.sql`)

## Example Questions for Agent

- "Predict service request volume for the next 6 months"
- "Which members are at high risk of cancellation?"
- "What is the predicted success rate for a towing request on Highway 101 during rain?"
- "Forecast demand for battery jump services next quarter"

The models will now be available as tools your agent can use!